<a href="https://colab.research.google.com/github/Hanna07111/masked-social-signals/blob/review-notes/code%20review/GPT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Blockwise Attention Mask

In [ ]:
class Attention(nn.Module):
    def __init__(self, nx, n_ctx, config, scale=False, is_cross_attention=False):
        super().__init__()

        n_state = nx  # in Attention: n_state=768 (nx=n_embd)
        # [switch nx => n_state from Block to Attention to keep identical to TF implem]
        assert n_state % config.n_head == 0

        # 블록크기(bundle) 설정
        self.bundle = config.n_bundle

        # 전체 길이 n_ctx가 bundle로 나누어 떨어져야 blockwise mask를 만들 수 있음 -> 확인하는 과정
        assert n_ctx % self.bundle == 0

        # strict causal mask(기본 하삼각 행렬부터)
        diy_causal_mask = torch.tril(torch.ones((n_ctx, n_ctx), dtype=torch.uint8), diagonal=-1)#.view(1, 1, n_ctx, n_ctx)

        # bundle 크기만큼 덮어씌울 필터를 만듦 -> 대각선만 0, 나머지 전부 1인 행렬
        filter_section = torch.ones((self.bundle, self.bundle), dtype=torch.uint8) - torch.eye(self.bundle, dtype=torch.uint8)

        # 각 블록의 영역을 filter_section으로 교체
        for i in range(0, n_ctx, self.bundle):
            diy_causal_mask[i:i+self.bundle, i:i+self.bundle] = filter_section
        diy_causal_mask = diy_causal_mask.view(1, 1, n_ctx, n_ctx)

        #import pdb; pdb.set_trace()
        self.register_buffer(
            "bias", diy_causal_mask
        )
        self.register_buffer("masked_bias", torch.tensor(-1e4))

### Right-Shifted Residual Connection

In [ ]:
class Block(nn.Module):
    def __init__(self, n_ctx, config, scale=False):
        super().__init__()
        hidden_size = config.n_embd
        inner_dim = config.n_inner if config.n_inner is not None else 4 * hidden_size
        self.ln_1 = nn.LayerNorm(hidden_size, eps=config.layer_norm_epsilon)
        self.attn = Attention(hidden_size, n_ctx, config, scale)
        self.ln_2 = nn.LayerNorm(hidden_size, eps=config.layer_norm_epsilon)
        # self.adapter_ln = nn.LayerNorm(hidden_size, eps=config.layer_norm_epsilon)
        if config.add_cross_attention:
            self.crossattention = Attention(hidden_size, n_ctx, config, scale, is_cross_attention=True)
            self.ln_cross_attn = nn.LayerNorm(hidden_size, eps=config.layer_norm_epsilon)
        self.mlp = MLP(inner_dim, config)
        # self.adapter_mlp = AdapterMLP(512, config)  # ADAPTER

    def forward(
            self,
            hidden_states,
            layer_past=None,
            attention_mask=None,
            head_mask=None,
            encoder_hidden_states=None,
            encoder_attention_mask=None,
            use_cache=False,
            output_attentions=False,
    ):
        attn_outputs = self.attn(
            self.ln_1(hidden_states),
            layer_past=layer_past,
            attention_mask=attention_mask,
            head_mask=head_mask,
            use_cache=use_cache,
            output_attentions=output_attentions,
        )
        attn_output = attn_outputs[0]  # output_attn: a, present, (attentions)
        outputs = attn_outputs[1:]

        # right shift the hidden_states

        # 첫 블록 길이 만큼 0으로 채운 텐서
        first = torch.zeros_like(hidden_states[:, :self.attn.bundle, :])

        # right shift -> 직전 블록의 표현 끌어옴
        right_shifted = torch.cat((first, hidden_states[:, :-self.attn.bundle, :]), dim=1)

        # residual connection
        # attention output과 right shifted 된 hidden state 값 더함
        hidden_states = attn_output  + right_shifted

        if encoder_hidden_states is not None:
            # add one self-attention block for cross-attention
            assert hasattr(
                self, "crossattention"
            ), f"If `encoder_hidden_states` are passed, {self} has to be instantiated with cross-attention layers by setting `config.add_cross_attention=True`"
            cross_attn_outputs = self.crossattention(
                self.ln_cross_attn(hidden_states),
                attention_mask=attention_mask,
                head_mask=head_mask,
                encoder_hidden_states=encoder_hidden_states,
                encoder_attention_mask=encoder_attention_mask,
                output_attentions=output_attentions,
            )
            attn_output = cross_attn_outputs[0]
            # residual connection
            hidden_states = hidden_states + attn_output
            outputs = outputs + cross_attn_outputs[2:]  # add cross attentions if we output attention weights

        feed_forward_hidden_states = self.mlp(self.ln_2(hidden_states))
        # residual connection
        hidden_states = hidden_states + feed_forward_hidden_states
        # hidden_states = hidden_states + self.adapter_ln(self.adapter_mlp(hidden_states))

        outputs = [hidden_states] + outputs
        return outputs  # hidden_states, present, (attentions, cross_attentions)